# 作业1：基于注意力的图像描述（ARCTIC）— PyTorch实现

**目标**：将PaddlePaddle版本的ARCTIC图像描述模型转换为PyTorch实现，在CPU上运行。

**要求**：
- 补全标有 `# TODO` 的代码块
- 所有代码基于PyTorch，不使用PaddlePaddle
- 默认在CPU上运行

**模型概述**：ARCTIC模型由CNN图像编码器（ResNet-101）和带有加性注意力机制的GRU解码器组成，使用Beam Search生成图像描述。

## 环境准备

In [ ]:
!pip install torch torchvision nltk pillow matplotlib -q

In [ ]:
import os
from os.path import join as pjoin
import json
import random
import numpy as np
from collections import defaultdict, Counter
from PIL import Image
from matplotlib import pyplot as plt
from argparse import Namespace

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

device = torch.device('cpu')
print(f'使用设备: {device}')
print(f'PyTorch版本: {torch.__version__}')

## 读取数据

以下数据处理代码已提供，无需修改。

In [ ]:
# 解压数据
# !unzip -q ./data/data243982/flickr8k.zip -d ./data/

In [ ]:
%matplotlib inline

def create_dataset(data_dir='./data', dataset='flickr8k',
                   captions_per_image=5, min_word_count=5, max_len=30):
    karpathy_json_path = pjoin(data_dir, '%s/dataset_flickr8k.json' % dataset)
    image_folder = pjoin(data_dir, '%s/images' % dataset)
    output_folder = pjoin(data_dir, '%s' % dataset)

    with open(karpathy_json_path, 'r') as j:
        data = json.load(j)

    image_paths = defaultdict(list)
    image_captions = defaultdict(list)
    vocab = Counter()

    for img in data['images']:
        split = img['split']
        captions = []
        for c in img['sentences']:
            if split != 'test':
                vocab.update(c['tokens'])
            if len(c['tokens']) <= max_len:
                captions.append(c['tokens'])
        if len(captions) == 0:
            continue
        path = os.path.join(image_folder, img['filename'])
        image_paths[split].append(path)
        image_captions[split].append(captions)

    words = [w for w in vocab.keys() if vocab[w] > min_word_count]
    vocab = {k: v + 1 for v, k in enumerate(words)}
    vocab['<pad>'] = 0
    vocab['<unk>'] = len(vocab)
    vocab['<start>'] = len(vocab)
    vocab['<end>'] = len(vocab)

    with open(os.path.join(output_folder, 'vocab.json'), 'w') as fw:
        json.dump(vocab, fw)

    for split in image_paths:
        imgpaths = image_paths[split]
        imcaps = image_captions[split]
        enc_captions = []
        for i, path in enumerate(imgpaths):
            img = Image.open(path)
            if len(imcaps[i]) < captions_per_image:
                captions = imcaps[i] + \
                    [random.choice(imcaps[i]) for _ in range(captions_per_image - len(imcaps[i]))]
            else:
                captions = random.sample(imcaps[i], k=captions_per_image)
            for j, c in enumerate(captions):
                enc_c = [vocab['<start>']] + [vocab.get(word, vocab['<unk>']) for word in c] + [vocab['<end>']]
                enc_captions.append(enc_c)
        data = {'IMAGES': imgpaths, 'CAPTIONS': enc_captions}
        with open(pjoin(output_folder, split + '_data.json'), 'w') as fw:
            json.dump(data, fw)

data_dir = './data'
if not os.path.exists(pjoin(data_dir, 'flickr8k', 'vocab.json')):
    create_dataset(data_dir)

### 任务1：定义数据集类（10分）

将PaddlePaddle的数据集类转换为PyTorch版本。

**提示**：
- `paddle.io.Dataset` → `torch.utils.data.Dataset`
- `paddle.to_tensor(x, dtype='int64')` → `torch.tensor(x, dtype=torch.long)`

In [ ]:
class ImageTextDataset(Dataset):
    def __init__(self, dataset_path, vocab_path, split, captions_per_image=5, max_len=30, transform=None):
        self.split = split
        assert self.split in {'train', 'val', 'test'}
        self.cpi = captions_per_image
        self.max_len = max_len

        with open(dataset_path, 'r') as f:
            self.data = json.load(f)
        with open(vocab_path, 'r') as f:
            self.vocab = json.load(f)

        self.transform = transform
        self.dataset_size = len(self.data['CAPTIONS'])

    def __getitem__(self, i):
        img = Image.open(self.data['IMAGES'][i // self.cpi]).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)

        caplen = len(self.data['CAPTIONS'][i])

        # TODO: 将caption转为PyTorch张量，并补齐到固定长度 (max_len + 2)
        # 提示：使用 torch.tensor()，dtype=torch.long
        # 补齐值为 self.vocab['<pad>']，总长度为 self.max_len + 2
        caption = None  # 请替换此行

        return img, caption, caplen

    def __len__(self):
        return self.dataset_size

In [ ]:
def mktrainval(data_dir, vocab_path, batch_size, workers=0):
    train_tx = transforms.Compose([
        transforms.Resize(256),
        transforms.RandomCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    val_tx = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    train_set = ImageTextDataset(os.path.join(data_dir, 'train_data.json'),
                                vocab_path, 'train', transform=train_tx)
    valid_set = ImageTextDataset(os.path.join(data_dir, 'val_data.json'),
                                vocab_path, 'val', transform=val_tx)
    test_set = ImageTextDataset(os.path.join(data_dir, 'test_data.json'),
                                vocab_path, 'test', transform=val_tx)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=workers)
    valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False, num_workers=workers, drop_last=False)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=workers, drop_last=False)

    return train_loader, valid_loader, test_loader

### 任务2：实现图像编码器（10分）

使用预训练ResNet-101提取图像特征。

**提示**：
- `paddle.vision.models.resnet101(pretrained=True)` → `models.resnet101(weights=models.ResNet101_Weights.DEFAULT)`
- 去掉最后两层（avgpool + fc），保留卷积特征
- `nn.Layer` → `nn.Module`

In [ ]:
class ImageEncoder(nn.Module):
    def __init__(self, finetuned=True):
        super(ImageEncoder, self).__init__()

        # TODO: 加载预训练的ResNet-101，去掉最后两层（avgpool和fc）
        # 提示：
        #   1. model = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        #   2. 使用 list(model.children())[:-2] 去掉最后两层
        #   3. 用 nn.Sequential(*...) 包装
        self.grid_representation_extractor = None  # 请替换此行

        # TODO: 设置参数是否需要微调
        # 提示：遍历 self.grid_representation_extractor.parameters()，设置 requires_grad
        pass  # 请补全

    def forward(self, images):
        # images: (batch, 3, 224, 224) -> out: (batch, 2048, 7, 7)
        out = self.grid_representation_extractor(images)
        return out

### 任务3：实现加性注意力机制（15分）

实现加性注意力（Additive Attention），计算查询和键值对之间的注意力权重。

**提示**：
- `nn.Softmax(axis=1)` → `nn.Softmax(dim=1)`
- `paddle.bmm()` → `torch.bmm()`

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, query_dim, key_dim, attn_dim):
        super(AdditiveAttention, self).__init__()
        self.attn_w_1_q = nn.Linear(query_dim, attn_dim)
        self.attn_w_1_k = nn.Linear(key_dim, attn_dim)
        self.attn_w_2 = nn.Linear(attn_dim, 1)
        self.tanh = nn.Tanh()
        self.softmax = nn.Softmax(dim=1)

    def forward(self, query, key_value):
        """
        参数：
            query: (batch_size, q_dim)
            key_value: (batch_size, n_kv, kv_dim)
        返回：
            output: (batch_size, kv_dim) 加权求和后的上下文向量
            attn: (batch_size, n_kv) 注意力权重
        """
        # TODO: 实现加性注意力的前向传播
        # 步骤：
        #   1. 将query映射到attn_dim: queries = self.attn_w_1_q(query).unsqueeze(1)
        #   2. 将key映射到attn_dim: keys = self.attn_w_1_k(key_value)
        #   3. 计算注意力得分: attn = self.attn_w_2(self.tanh(queries + keys)).squeeze(2)
        #   4. 归一化: attn = self.softmax(attn)
        #   5. 加权求和: output = torch.bmm(attn.unsqueeze(1), key_value).squeeze(1)

        output = None  # 请替换
        attn = None    # 请替换

        return output, attn

### 任务4：实现注意力解码器（20分）

实现带注意力机制的GRU解码器。这是本作业的核心部分。

**提示**：
- `nn.initializer.Uniform(low=-0.1, high=0.1)` → `nn.init.uniform_(tensor, -0.1, 0.1)`
- Paddle GRU `time_major=True` → PyTorch GRU `batch_first=False`（默认值）
- `paddle.concat` → `torch.cat`
- `paddle.zeros` → `torch.zeros`
- 隐状态reshape后需调用 `.contiguous()`
- 排序：`(-cap_lens).argsort(axis=0)` → `cap_lens.argsort(descending=True)`

In [ ]:
class AttentionDecoder(nn.Module):
    def __init__(self, image_code_dim, vocab_size, word_dim, attention_dim, hidden_size, num_layers, dropout=0.5):
        super(AttentionDecoder, self).__init__()

        # TODO: 初始化嵌入层，并用均匀分布 [-0.1, 0.1] 初始化权重
        # 提示：
        #   self.embed = nn.Embedding(vocab_size, word_dim)
        #   nn.init.uniform_(self.embed.weight, -0.1, 0.1)
        self.embed = None  # 请替换

        self.attention = AdditiveAttention(hidden_size, image_code_dim, attention_dim)

        # TODO: 初始化隐状态映射层，用均匀分布初始化
        self.init_state = None  # 请替换

        # TODO: 初始化GRU
        # 提示：nn.GRU(input_size=word_dim + image_code_dim, hidden_size=hidden_size, num_layers=num_layers)
        # 注意：PyTorch默认 batch_first=False，输入shape为 (seq_len, batch, input_size)
        self.rnn = None  # 请替换

        self.dropout = nn.Dropout(p=dropout)

        # TODO: 初始化输出全连接层 hidden_size -> vocab_size，用均匀分布初始化
        self.fc = None  # 请替换

        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.vocab_size = vocab_size

    def init_hidden_state(self, image_code, captions, cap_lens):
        """初始化隐状态，并按caption长度降序排列"""
        batch_size, image_code_dim = image_code.shape[0], image_code.shape[1]

        # TODO: 将image_code从 (batch, C, H, W) 转换为 (batch, H*W, C)
        # 提示：先 .permute(0, 2, 3, 1)，再 .reshape(batch_size, -1, image_code_dim)
        image_code = None  # 请替换

        # TODO: 按caption长度从长到短排序
        # 提示：sorted_cap_indices = cap_lens.argsort(descending=True)
        sorted_cap_indices = None  # 请替换
        sorted_cap_lens = cap_lens[sorted_cap_indices]
        captions = captions[sorted_cap_indices]
        image_code = image_code[sorted_cap_indices]

        if captions.dim() == 1:
            captions = captions.unsqueeze(0)
        if image_code.dim() == 2:
            image_code = image_code.unsqueeze(0)

        # TODO: 用图像特征的均值初始化隐状态
        # 提示：
        #   hidden_state = self.init_state(image_code.mean(dim=1))
        #   然后reshape为 (num_layers, batch_size, hidden_size)，并调用 .contiguous()
        hidden_state = None  # 请替换

        return image_code, captions, sorted_cap_lens, sorted_cap_indices, hidden_state

    def forward_step(self, image_code, curr_cap_embed, hidden_state):
        """解码器单步前向传播"""
        # TODO: 实现单步解码
        # 步骤：
        #   1. 用注意力机制获取上下文向量: context, alpha = self.attention(hidden_state[-1], image_code)
        #   2. 拼接上下文和当前词嵌入: x = torch.cat((context, curr_cap_embed), dim=-1).unsqueeze(0)
        #   3. 通过GRU: out, hidden_state = self.rnn(x, hidden_state)
        #   4. 预测: preds = self.fc(self.dropout(out.squeeze(0)))

        preds = None       # 请替换
        alpha = None       # 请替换
        hidden_state = None  # 请替换

        return preds, alpha, hidden_state

    def forward(self, image_code, captions, cap_lens):
        """完整前向传播（训练时使用）"""
        image_code, captions, sorted_cap_lens, sorted_cap_indices, hidden_state = \
            self.init_hidden_state(image_code, captions, cap_lens)
        batch_size = image_code.shape[0]
        lengths = sorted_cap_lens.numpy() - 1

        # TODO: 初始化predictions和alphas张量
        # 提示：
        #   predictions = torch.zeros(batch_size, lengths[0], self.vocab_size)
        #   alphas = torch.zeros(batch_size, lengths[0], image_code.shape[1])
        predictions = None  # 请替换
        alphas = None       # 请替换

        cap_embeds = self.embed(captions)

        for step in range(lengths[0]):
            real_batch_size = np.where(lengths > step)[0].shape[0]
            preds, alpha, hidden_state = self.forward_step(
                image_code[:real_batch_size],
                cap_embeds[:real_batch_size, step, :],
                hidden_state[:, :real_batch_size, :].contiguous()
            )
            predictions[:real_batch_size, step, :] = preds
            alphas[:real_batch_size, step, :] = alpha

        return predictions, alphas, captions, lengths, sorted_cap_indices

### 任务5：实现ARCTIC模型和Beam Search（15分）

组合编码器和解码器，并实现Beam Search解码。

**提示**：
- `paddle.full(shape, val)` → `torch.full(shape, val, dtype=torch.long)`
- `repeat_interleave(x, k, dim=0)` 用法相同
- `F.log_softmax(preds, dim=1)` 需要显式指定dim

In [ ]:
class ARCTIC(nn.Module):
    def __init__(self, image_code_dim, vocab, word_dim, attention_dim, hidden_size, num_layers):
        super(ARCTIC, self).__init__()
        self.vocab = vocab
        self.encoder = ImageEncoder()
        self.decoder = AttentionDecoder(image_code_dim, len(vocab), word_dim, attention_dim, hidden_size, num_layers)

    def forward(self, images, captions, cap_lens):
        image_code = self.encoder(images)
        return self.decoder(image_code, captions, cap_lens)

    def generate_by_beamsearch(self, images, beam_k, max_len):
        """使用Beam Search生成图像描述"""
        vocab_size = len(self.vocab)
        image_codes = self.encoder(images)
        texts = []

        for image_code in image_codes:
            # TODO: 实现Beam Search
            # 步骤：
            #   1. 复制image_code为beam_k份: image_code.unsqueeze(0).repeat_interleave(beam_k, dim=0)
            #   2. 初始化当前句子: torch.full((beam_k, 1), self.vocab['<start>'], dtype=torch.long)
            #   3. 初始化隐状态
            #   4. 循环：
            #      a. 单步解码并计算log概率: F.log_softmax(preds, dim=1)
            #      b. 累加概率并选top-k
            #      c. 处理<end>标记：将完成的句子保存
            #      d. 更新状态继续生成
            #   5. 选择概率最高的句子

            image_code = image_code.unsqueeze(0).repeat_interleave(beam_k, dim=0)
            cur_sents = torch.full((beam_k, 1), self.vocab['<start>'], dtype=torch.long)
            cur_sent_embed = self.decoder.embed(cur_sents)[:, 0, :]
            sent_lens = torch.ones(beam_k, dtype=torch.long)
            image_code, cur_sent_embed, _, _, hidden_state = \
                self.decoder.init_hidden_state(image_code, cur_sent_embed, sent_lens)

            end_sents = []
            end_probs = []
            probs = torch.zeros(beam_k, 1)
            k = beam_k

            while True:
                preds, _, hidden_state = self.decoder.forward_step(
                    image_code[:k], cur_sent_embed, hidden_state)

                # TODO: 计算log概率并累加
                # 提示：preds = F.log_softmax(preds, dim=1)
                #        probs = probs.expand_as(preds) + preds
                pass  # 请补全

                # TODO: 选择top-k
                # 提示：
                #   首次（cur_sents.shape[1]==1）从第一行选top-k
                #   之后从展平的probs中选top-k
                #   sent_indices = indices // vocab_size
                #   word_indices = indices % vocab_size
                pass  # 请补全

                # TODO: 更新句子，处理<end>标记
                # 提示：
                #   cur_sents = torch.cat([cur_sents[sent_indices], word_indices.unsqueeze(1)], dim=1)
                #   检查word_indices中哪些等于self.vocab['<end>']
                pass  # 请补全

                if cur_sents.shape[1] >= max_len:
                    break

            if len(end_sents) == 0:
                gen_sent = cur_sents[0].tolist()
            else:
                gen_sent = end_sents[end_probs.index(max(end_probs))]
            texts.append(gen_sent)
        return texts

### 任务6：定义损失函数（5分）

**提示**：
- `nn.Layer` → `nn.Module`
- `paddle.concat` → `torch.cat`

In [ ]:
class CrossEntropyLoss(nn.Module):
    def __init__(self):
        super(CrossEntropyLoss, self).__init__()
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, predictions, targets, lengths):
        # TODO: 将变长的predictions和targets拼接后计算交叉熵损失
        # 步骤：
        #   1. 遍历每个样本，按实际长度截取 predictions[i, :lengths[i], :] 和 targets[i, :lengths[i]]
        #   2. 用 torch.cat 拼接所有样本
        #   3. 调用 self.loss_fn 计算损失

        loss = None  # 请替换
        return loss

In [ ]:
def get_optimizer(model, config):
    return torch.optim.Adam(model.parameters(), lr=config.learning_rate)

### 任务7：实现评估函数（5分）

**提示**：
- 使用 `torch.no_grad()` 上下文管理器禁用梯度计算
- `model.eval()` / `model.train()` 切换模式

In [ ]:
from nltk.translate.bleu_score import corpus_bleu

def filter_useless_words(sent, filterd_words):
    return [w for w in sent if w not in filterd_words]

def evaluate(data_loader, model, config):
    # TODO: 实现评估函数
    # 步骤：
    #   1. model.eval() 切换到评估模式
    #   2. 在 torch.no_grad() 下遍历数据
    #   3. 调用 model.generate_by_beamsearch 生成描述
    #   4. 计算 corpus_bleu (BLEU-4)
    #   5. model.train() 切回训练模式

    bleu4 = 0.0  # 请替换
    return bleu4

### 任务8：完成训练循环（20分）

**提示**：
- `optimizer.clear_grad()` → `optimizer.zero_grad()`
- `nn.utils.clip_grad_norm_()` 用法相同
- `paddle.save()` → `torch.save()`
- `loss.item()` 获取标量值

In [ ]:
config = Namespace(
    max_len=30,
    captions_per_image=5,
    batch_size=32,
    image_code_dim=2048,
    word_dim=512,
    hidden_size=512,
    attention_dim=512,
    num_layers=1,
    learning_rate=0.0005,
    num_epochs=10,
    grad_clip=5.0,
    alpha_weight=1.0,
    evaluate_step=900,
    checkpoint=None,
    best_checkpoint='model/ARCTIC/best_flickr8k.ckpt',
    last_checkpoint='model/ARCTIC/last_flickr8k.ckpt',
    beam_k=5
)

data_dir = './data'
vocab_path = pjoin(data_dir, 'flickr8k/vocab.json')
train_loader, valid_loader, test_loader = mktrainval(
    pjoin(data_dir, 'flickr8k'), vocab_path, config.batch_size)

with open(vocab_path, 'r') as f:
    vocab = json.load(f)

model = ARCTIC(config.image_code_dim, vocab, config.word_dim,
               config.attention_dim, config.hidden_size, config.num_layers)
model = model.to(device)

optimizer = get_optimizer(model, config)
loss_fn = CrossEntropyLoss()

os.makedirs(os.path.dirname(config.best_checkpoint), exist_ok=True)
model.train()
best_res = 0
print("开始训练")

for epoch in range(config.num_epochs):
    for i, (imgs, caps, caplens) in enumerate(train_loader):
        imgs = imgs.to(device)
        caps = caps.to(device)
        caplens = torch.tensor(caplens, dtype=torch.long)

        # TODO: 完成训练步骤
        # 步骤：
        #   1. optimizer.zero_grad()  — 清零梯度
        #   2. 前馈: predictions, alphas, sorted_captions, lengths, sorted_cap_indices = model(imgs, caps, caplens)
        #   3. 计算损失: loss = loss_fn(predictions, sorted_captions[:, 1:], lengths)
        #              loss += config.alpha_weight * ((1. - alphas.sum(dim=1)) ** 2).mean()
        #   4. loss.backward()  — 反向传播
        #   5. nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)  — 梯度裁剪
        #   6. optimizer.step()  — 更新参数

        pass  # 请补全

        if (i + 1) % 100 == 0:
            print(f'epoch {epoch}, step {i+1}: loss={loss.item():.2f}')

        if (i + 1) % config.evaluate_step == 0:
            bleu_score = evaluate(valid_loader, model, config)
            state = {
                'epoch': epoch, 'step': i,
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict()
            }
            if best_res < bleu_score:
                best_res = bleu_score
                torch.save(state, config.best_checkpoint)
            torch.save(state, config.last_checkpoint)
            print(f'Validation@epoch {epoch}, step {i+1}, BLEU-4={bleu_score:.4f}')

## Paddle → PyTorch 对照速查表

| Paddle | PyTorch | 说明 |
|--------|---------|------|
| `nn.Layer` | `nn.Module` | 模型基类 |
| `paddle.to_tensor(x, dtype='int64')` | `torch.tensor(x, dtype=torch.long)` | 创建张量 |
| `paddle.concat(tensors, axis=0)` | `torch.cat(tensors, dim=0)` | 拼接，`axis`→`dim` |
| `paddle.zeros / full` | `torch.zeros / full` | 创建张量 |
| `paddle.bmm(a, b)` | `torch.bmm(a, b)` | 批矩阵乘 |
| `tensor.transpose(perm)` | `tensor.permute(*perm)` | 维度变换 |
| `Softmax(axis=1)` | `Softmax(dim=1)` | `axis`→`dim` |
| `optimizer.clear_grad()` | `optimizer.zero_grad()` | 清除梯度 |
| `paddle.save / load` | `torch.save / load` | 保存/加载 |
| `nn.initializer.Uniform` | `nn.init.uniform_` | 均匀初始化 |
| `paddle.io.Dataset` | `torch.utils.data.Dataset` | 数据集基类 |
| `paddle.vision.models` | `torchvision.models` | 预训练模型 |
| GRU `time_major=True` | GRU `batch_first=False` | 时序维度在前 |